In [1]:
# =============================================================================
# CELL 1 — Install Dependencies
# =============================================================================
# Purpose : Install requests for EIA and Open-Meteo HTTP calls.
# =============================================================================

%pip install requests --quiet

print("✅ Dependencies installed")

StatementMeta(, 313d22de-077d-4a73-a53d-99251ef0197f, 8, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed



In [2]:
# =============================================================================
# CELL 2 — Imports & Configuration
# =============================================================================
# Purpose : Load credentials and configure the 30-minute weather/EIA poller.
#
# Schedule : Every 30 minutes via Fabric Pipeline
#
# Data fetched:
#   - Visual Crossing : temperature, wind speed, humidity, solar radiation
#                       for 20 cities across all monitored regions
#                       (Open-Meteo blocked on Fabric Trial network)
#   - EIA             : hourly electricity demand for 13 US RTOs
#
# Rate limits:
#   - Visual Crossing : 1,000 records/day free — we use 960 (96%)
#   - EIA             : Conservative ~624 calls/day
# =============================================================================

import time
import random
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)

# -----------------------------------------------------------------------------
# API Credentials — from Fabric Environment
# -----------------------------------------------------------------------------
EIA_API_KEY = spark.conf.get("spark.pulsegrid.eia_key")
VC_API_KEY  = spark.conf.get("spark.pulsegrid.visualcrossing_key")
print(f"✅ EIA key loaded : {EIA_API_KEY[:8]}...{EIA_API_KEY[-4:]}")
print(f"✅ VC key loaded  : {VC_API_KEY[:8]}...{VC_API_KEY[-4:]}")

# -----------------------------------------------------------------------------
# Rate Limiting
# -----------------------------------------------------------------------------
MAX_RETRIES    = 3
BACKOFF_FACTOR = 2
JITTER_MAX     = 1.0

# -----------------------------------------------------------------------------
# KQL Bronze Target
# -----------------------------------------------------------------------------
KUSTO_CLUSTER  = "https://trd-ratdj1p1b0yurnmn41.z4.kusto.fabric.microsoft.com"
KUSTO_DATABASE = "pulsegrid_bronze"

# -----------------------------------------------------------------------------
# Visual Crossing — 20 cities (reduced from 30 to stay within 1,000/day limit)
# 20 cities × 48 cycles/day = 960 calls/day = 96% of free tier
# -----------------------------------------------------------------------------
WEATHER_CITIES = {
    "DE-Berlin"    : {"lat": 52.52,  "lon": 13.41,  "region": "DE-LU"},
    "FR-Paris"     : {"lat": 48.85,  "lon": 2.35,   "region": "FR"},
    "ES-Madrid"    : {"lat": 40.42,  "lon": -3.70,  "region": "ES"},
    "NL-Amsterdam" : {"lat": 52.37,  "lon": 4.90,   "region": "NL"},
    "BE-Brussels"  : {"lat": 50.85,  "lon": 4.35,   "region": "BE"},
    "PL-Warsaw"    : {"lat": 52.23,  "lon": 21.01,  "region": "PL"},
    "AT-Vienna"    : {"lat": 48.21,  "lon": 16.37,  "region": "AT"},
    "CH-Zurich"    : {"lat": 47.38,  "lon": 8.54,   "region": "CH"},
    "PT-Lisbon"    : {"lat": 38.72,  "lon": -9.14,  "region": "PT"},
    "FI-Helsinki"  : {"lat": 60.17,  "lon": 24.94,  "region": "FI"},
    "IT-Milan"     : {"lat": 45.46,  "lon": 9.19,   "region": "IT-NO"},
    "DK-Copenhagen": {"lat": 55.68,  "lon": 12.57,  "region": "DK-2"},
    "SE-Stockholm" : {"lat": 59.33,  "lon": 18.07,  "region": "SE-3"},
    "NO-Oslo"      : {"lat": 59.91,  "lon": 10.75,  "region": "NO-2"},
    "GR-Athens"    : {"lat": 37.98,  "lon": 23.73,  "region": "GR"},
    "US-Houston"   : {"lat": 29.76,  "lon": -95.37, "region": "US-ERCOT"},
    "US-Chicago"   : {"lat": 41.88,  "lon": -87.63, "region": "US-MISO"},
    "US-NewYork"   : {"lat": 40.71,  "lon": -74.01, "region": "US-NYIS"},
    "US-Boston"    : {"lat": 42.36,  "lon": -71.06, "region": "US-ISNE"},
    "US-LosAngeles": {"lat": 34.05,  "lon": -118.24,"region": "US-CISO"},
}

# -----------------------------------------------------------------------------
# EIA — 13 US RTOs
# -----------------------------------------------------------------------------
EIA_REGIONS = ["ERCO", "MISO", "PJM", "NYIS", "ISNE", "CISO",
               "SWPP", "BPAT", "PACW", "PACE", "NEVP", "SRP", "APS"]

EIA_REGION_MAP = {
    "ERCO": "US-ERCOT", "MISO": "US-MISO", "PJM": "US-PJM",
    "NYIS": "US-NYIS",  "ISNE": "US-ISNE", "CISO": "US-CISO",
    "SWPP": "US-SWPP",  "BPAT": "US-BPAT", "PACW": "US-PACW",
    "PACE": "US-PACE",  "NEVP": "US-NEVP", "SRP":  "US-SRP",
    "APS":  "US-APS"
}

print(f"✅ Config loaded")
print(f"   Weather cities  : {len(WEATHER_CITIES)}")
print(f"   EIA regions     : {len(EIA_REGIONS)}")

StatementMeta(, 313d22de-077d-4a73-a53d-99251ef0197f, 10, Finished, Available, Finished, False)

✅ EIA key loaded : Yz2G8N3b...1td4
✅ VC key loaded  : HDHRUGPP...A7C6
✅ Config loaded
   Weather cities  : 20
   EIA regions     : 13


In [3]:
# =============================================================================
# CELL 3 — Retry Helper + Generic KQL Writer
# =============================================================================
# Purpose : Provide shared utilities used by both weather and EIA fetchers.
#
# call_with_retry:
#   Wraps any API call with exponential backoff + jitter.
#   Attempt 1 fail → wait 2^1 + jitter ≈ 2-3s
#   Attempt 2 fail → wait 2^2 + jitter ≈ 4-5s
#   Attempt 3 fail → wait 2^3 + jitter ≈ 8-9s
#   Jitter prevents thundering herd on concurrent pipeline triggers.
#
# records_to_spark:
#   Converts list of dicts → Spark DataFrame.
#   Arrow optimization disabled — prevents BufferHolder negative size error
#   caused by mixed None + numeric types in pandas columns (same fix as 01a).
#
# write_to_kql:
#   Appends Spark DataFrame to Bronze KQL table.
#   Append-only — Bronze is an immutable raw store, no updates or deletes.
# =============================================================================

def call_with_retry(fn, *args, **kwargs):
    """Exponential backoff retry wrapper."""
    last_exception = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_exception = e
            wait = (BACKOFF_FACTOR ** attempt) + random.uniform(0, JITTER_MAX)
            print(f"   ⚠️  Attempt {attempt}/{MAX_RETRIES} failed: {type(e).__name__} — retrying in {wait:.1f}s")
            time.sleep(wait)
    raise last_exception


def records_to_spark(records, schema):
    """
    Convert list of dicts to Spark DataFrame.
    Arrow disabled to handle nullable float64 columns safely.
    """
    pdf = pd.DataFrame(records)
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
    df  = spark.createDataFrame(pdf, schema=schema)
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    return df


def write_to_kql(df_spark, table_name):
    """
    Append Spark DataFrame to Bronze KQL table.
    Uses managed identity token — no hardcoded credentials.
    """
    df_spark.write \
        .format("com.microsoft.kusto.spark.datasource") \
        .option("kustoCluster",  KUSTO_CLUSTER) \
        .option("kustoDatabase", KUSTO_DATABASE) \
        .option("kustoTable",    table_name) \
        .option("accessToken",   mssparkutils.credentials.getToken("kusto")) \
        .mode("append") \
        .save()

print("✅ Retry helper + KQL writer defined")

StatementMeta(, 313d22de-077d-4a73-a53d-99251ef0197f, 11, Finished, Available, Finished, False)

✅ Retry helper + KQL writer defined


In [4]:
# =============================================================================
# CELL 3b — Network Connectivity Diagnostic
# =============================================================================
# Purpose : Verify outbound HTTP access from Fabric Spark to external APIs.
#           Run this cell manually to diagnose connectivity issues.
#           Not part of the scheduled pipeline execution — diagnostic only.
#
# Expected results:
#   ✅ EIA reachable        — Fabric allows outbound to api.eia.gov
#   ❌ Open-Meteo blocked   — Fabric Trial restricts api.open-meteo.com
#                             Weather features derived from generation mix in Gold
# =============================================================================

import requests

test_urls = [
    ("Open-Meteo", "https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&current=temperature_2m"),
    ("EIA",        "https://api.eia.gov/v2/electricity/rto/region-data/data/?api_key=Yz2G8N3bCq6gbaRyXkkloltlYSt0fLq52BWJ1td4&frequency=hourly&data[0]=value&facets[respondent][]=ERCO&facets[type][]=D&length=1"),
]

for name, url in test_urls:
    try:
        resp = requests.get(url, timeout=30)
        print(f"✅ {name} reachable — status {resp.status_code}")
    except requests.exceptions.ReadTimeout:
        print(f"❌ {name} — ReadTimeout (blocked or too slow)")
    except requests.exceptions.ConnectionError as e:
        print(f"❌ {name} — ConnectionError: {e}")
    except Exception as e:
        print(f"❌ {name} — {type(e).__name__}: {e}")

StatementMeta(, 313d22de-077d-4a73-a53d-99251ef0197f, 12, Finished, Available, Finished, False)

✅ Open-Meteo reachable — status 200
✅ EIA reachable — status 200


In [5]:
# =============================================================================
# CELL 4 — Visual Crossing Weather Fetcher
# =============================================================================
# Purpose : Fetch current weather for 20 cities using Visual Crossing API.
#           Replaces Open-Meteo which is blocked on Fabric Spark network.
#
# API Details:
#   - Endpoint  : weather.visualcrossing.com Timeline API
#   - Variables : temp (°C), windspeed (km/h → converted to m/s),
#                 humidity (%), solarradiation (W/m²)
#   - Auth      : API key from Fabric Environment (spark.pulsegrid.visualcrossing_key)
#   - Free tier : 1,000 records/day, commercial use allowed
#
# Rate limiting:
#   - 20 cities × 48 cycles/day = 960 calls/day (96% of free limit)
#   - 0.5s inter-request delay — conservative
#   - Windspeed converted from km/h to m/s (/3.6) for consistency with schema
# =============================================================================

WEATHER_SCHEMA = StructType([
    StructField("ingestion_time",  TimestampType(), False),
    StructField("event_time",      TimestampType(), False),
    StructField("region",          StringType(),    False),
    StructField("temperature_c",   DoubleType(),    True),
    StructField("wind_speed_ms",   DoubleType(),    True),
    StructField("humidity_pct",    DoubleType(),    True),
    StructField("solar_radiation", DoubleType(),    True),
    StructField("source",          StringType(),    True),
])

def fetch_weather_city(city_name, coords):
    """Fetch current weather for one city via Visual Crossing Timeline API."""
    location = f"{coords['lat']},{coords['lon']}"
    url = (
        f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/"
        f"services/timeline/{location}/today"
    )
    params = {
        "unitGroup"   : "metric",
        "elements"    : "temp,windspeed,humidity,solarradiation",
        "include"     : "current",
        "key"         : VC_API_KEY,
        "contentType" : "json"
    }
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    curr = data.get("currentConditions", {})

    return {
        "ingestion_time" : datetime.now(timezone.utc),
        "event_time"     : datetime.now(timezone.utc),
        "region"         : coords["region"],
        "temperature_c"  : float(curr["temp"])            if curr.get("temp")            is not None else None,
        "wind_speed_ms"  : float(curr["windspeed"]) / 3.6 if curr.get("windspeed")       is not None else None,
        "humidity_pct"   : float(curr["humidity"])        if curr.get("humidity")        is not None else None,
        "solar_radiation": float(curr["solarradiation"])  if curr.get("solarradiation")  is not None else None,
        "source"         : "VisualCrossing",
    }


def fetch_all_weather():
    """Fetch weather for all 20 cities."""
    all_records = []
    failed      = []

    for city_name, coords in WEATHER_CITIES.items():
        try:
            record = call_with_retry(fetch_weather_city, city_name, coords)
            all_records.append(record)
            time.sleep(0.5 + random.uniform(0, 0.3))
        except Exception as e:
            failed.append(city_name)
            print(f"   ⚠️  {city_name} failed: {e}")

    print(f"   ✅ Weather records  : {len(all_records)}")
    print(f"      Failed cities    : {failed if failed else 'None'}")
    return all_records

print("✅ Visual Crossing weather fetcher defined — 20 cities, 960 calls/day")

StatementMeta(, 313d22de-077d-4a73-a53d-99251ef0197f, 13, Finished, Available, Finished, False)

✅ Visual Crossing weather fetcher defined — 20 cities, 960 calls/day


In [6]:
# =============================================================================
# CELL 5 — EIA US Electricity Demand Fetcher
# =============================================================================
# Purpose : Fetch latest hourly electricity demand for 13 US RTOs.
#           One API call per region — returns last 3 hourly records.
#
# Rate limiting:
#   - 13 calls per cycle × 48 cycles/day = 624 calls/day
#   - EIA limit: unpublished but conservative usage applies
#   - 1s inter-request delay
# =============================================================================

PRICES_SCHEMA = StructType([
    StructField("ingestion_time", TimestampType(), False),
    StructField("event_time",     TimestampType(), False),
    StructField("region",         StringType(),    False),
    StructField("price_eur_mwh",  DoubleType(),    True),
    StructField("load_mw",        DoubleType(),    True),
    StructField("temperature_c",  DoubleType(),    True),
    StructField("source",         StringType(),    True),
])

def fetch_eia_region(respondent):
    """Fetch latest hourly demand for one EIA region."""
    url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"
    params = {
        "api_key"               : EIA_API_KEY,
        "frequency"             : "hourly",
        "data[0]"               : "value",
        "facets[respondent][]"  : respondent,
        "facets[type][]"        : "D",
        "sort[0][column]"       : "period",
        "sort[0][direction]"    : "desc",
        "length"                : 3
    }
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()

    rows    = resp.json().get("response", {}).get("data", [])
    records = []
    ingestion_time = datetime.now(timezone.utc)

    for row in rows:
        try:
            event_time = datetime.strptime(
                row["period"], "%Y-%m-%dT%H"
            ).replace(tzinfo=timezone.utc)
        except:
            continue

        val = row.get("value")
        records.append({
            "ingestion_time" : ingestion_time,
            "event_time"     : event_time,
            "region"         : EIA_REGION_MAP.get(respondent, f"US-{respondent}"),
            "price_eur_mwh"  : None,           # EIA demand data has no price
            "load_mw"        : float(val) if val is not None else None,
            "temperature_c"  : None,
            "source"         : f"EIA-{respondent}",
        })
    return records


def fetch_all_eia():
    """Fetch demand data for all 13 EIA regions."""
    all_records = []
    failed      = []

    for respondent in EIA_REGIONS:
        try:
            records = call_with_retry(fetch_eia_region, respondent)
            all_records.extend(records)
            time.sleep(1.0 + random.uniform(0, 0.5))
        except Exception as e:
            failed.append(respondent)

    print(f"   ✅ EIA records    : {len(all_records)}")
    print(f"      Failed regions : {failed if failed else 'None'}")
    return all_records

print("✅ EIA fetcher defined")

StatementMeta(, 313d22de-077d-4a73-a53d-99251ef0197f, 14, Finished, Available, Finished, False)

✅ EIA fetcher defined


In [7]:
# =============================================================================
# CELL 6 — Main Execution (Weather + EIA)
# =============================================================================
# Purpose : Run full weather + EIA poll cycle and write both to Bronze.
#           Weather via Visual Crossing (replaces blocked Open-Meteo).
#           EIA demand for 13 US RTOs written to raw_electricity_prices.
# =============================================================================

cycle_start = datetime.now(timezone.utc)
print("=" * 55)
print(f"  PulseGrid Weather/EIA Poller — {cycle_start.strftime('%Y-%m-%d %H:%M UTC')}")
print("=" * 55)

# ------------------------------------------------------------------
# Step 1 — Visual Crossing Weather (20 cities)
# ------------------------------------------------------------------
print("\n🌡️  Fetching weather data (20 cities via Visual Crossing)...")
weather_records = fetch_all_weather()

if weather_records:
    df_weather = records_to_spark(weather_records, WEATHER_SCHEMA)
    write_to_kql(df_weather, "raw_weather")
    print(f"✅ Weather written: {len(weather_records)} records")
else:
    print("⚠️  No weather records")

# ------------------------------------------------------------------
# Step 2 — EIA Demand (13 RTOs)
# ------------------------------------------------------------------
print("\n📡 Fetching EIA demand data (13 RTOs)...")
eia_records = fetch_all_eia()

if eia_records:
    pdf = pd.DataFrame(eia_records)
    pdf["ingestion_time"] = pd.to_datetime(pdf["ingestion_time"], utc=True)
    pdf["event_time"]     = pd.to_datetime(pdf["event_time"],     utc=True)
    pdf["price_eur_mwh"]  = pd.to_numeric(pdf["price_eur_mwh"],  errors="coerce").astype("float64")
    pdf["load_mw"]        = pd.to_numeric(pdf["load_mw"],        errors="coerce").astype("float64")
    pdf["temperature_c"]  = pd.to_numeric(pdf["temperature_c"],  errors="coerce").astype("float64")

    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
    df_eia = spark.createDataFrame(pdf, schema=PRICES_SCHEMA)
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

    write_to_kql(df_eia, "raw_electricity_prices")
    print(f"✅ EIA demand written: {len(eia_records)} records")
else:
    print("⚠️  No EIA records")

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
duration = (datetime.now(timezone.utc) - cycle_start).total_seconds()
print(f"\n{'='*55}")
print(f"  Cycle complete — {duration:.1f}s")
print(f"  Weather records : {len(weather_records)}")
print(f"  EIA records     : {len(eia_records)}")
print(f"{'='*55}")

StatementMeta(, 313d22de-077d-4a73-a53d-99251ef0197f, 15, Finished, Available, Finished, False)

  PulseGrid Weather/EIA Poller — 2026-08-14 08:29 UTC

🌡️  Fetching weather data (20 cities via Visual Crossing)...
   ✅ Weather records  : 20
      Failed cities    : None
✅ Weather written: 20 records

📡 Fetching EIA demand data (13 RTOs)...
   ✅ EIA records    : 36
      Failed regions : None
✅ EIA demand written: 36 records

  Cycle complete — 220.2s
  Weather records : 20
  EIA records     : 36
